# 演習8 解答編 ―― 並列化すると順序が崩れる

> まず `ex08_ordering.ipynb` を自分で解いてから読んでください。

## 発展課題1 の解答 ―― 預かる枚数は何で決まるか

**「どれだけ追い越されうるか」**で決まります。そしてそれは、
**同時に処理中のフレーム数**、つまり担当者の人数でほぼ決まります。

3人が同時に処理しているなら、いちばん若い番号が終わるまでのあいだに、
先に終わりうるのは残り2人ぶんです。加えて、キューに待っているぶんも預かりに乗ります。

次のセルで、人数を変えて測ります。

In [ ]:
%%writefile ans08a.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <map>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（ここでは中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 30;
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }
int infer_ms(int i) { return 20 + (i % 3) * 40; }   // 20 / 60 / 100、平均 60

void run(int ninfer) {
    BoundedQueue<int> q1(4), q2(4);
    std::size_t worst = 0;
    auto t0 = steady_clock::now();

    std::thread reader([&] { for (int i = 0; i < N; i++) { wait_ms(10); q1.push(i); } });
    std::vector<std::thread> inferers;
    for (int k = 0; k < ninfer; k++)
        inferers.emplace_back([&, k] {
            for (int i = k; i < N; i += ninfer) { int f = q1.pop(); wait_ms(infer_ms(f)); q2.push(f); }
        });

    std::thread shower([&] {
        std::map<int, int> pending; int next = 0;
        for (int i = 0; i < N; i++) {
            pending[q2.pop()] = 1;
            if (pending.size() > worst) worst = pending.size();
            while (!pending.empty() && pending.begin()->first == next) {
                pending.erase(pending.begin()); wait_ms(30); next++;
            }
        }
    });
    reader.join();
    for (auto& t : inferers) t.join();
    shower.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - t0).count();
    std::cout << std::right << std::setw(8) << ninfer << std::setw(12) << ms
              << std::setw(10) << std::fixed << std::setprecision(1) << (1000.0 * N / ms)
              << std::setw(14) << worst << "\n";
}

int main() {
    std::cout << "Infer の人数を変えて、並べ直しのために預かる最大数を見る\n\n";
    std::cout << "  Infer人数   time(ms)      FPS   預かる最大数\n";
    std::cout << "-------------------------------------------------\n";
    for (int n : {1, 2, 3, 4, 6}) run(n);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans08a.cpp -o ans08a && ./ans08a

```
  Infer人数   time(ms)      FPS   預かる最大数
       1        1846      16.3             1
       2        1063      28.2             3
       3         976      30.7             3
       4         976      30.7             4
       6         976      30.7             5
```

- **1人 … 預かりは 1。** 追い越しが起きないので、並べ直しは何もしていません
- **人数を増やすほど、預かりも増えます**

そして注目すべきは、**3人から先は FPS が伸びていない**ことです。
ボトルネックが Show（30ms）に移っているからです（演習7）。

つまり 4人・6人は、**速くならないのに預かりだけが増えている**状態です。

> **人を増やしすぎると、速くならないうえに、順序の崩れとメモリだけが増える。**

増やす人数を決めるときは、FPS だけでなく**この列も見る**べきです。

## 発展課題2 の解答 ―― フレームが二度と来ないとき

**そこで静かに止まります。**

`next` 番が来るまで、後続はいくら届いても出せません。
預かり場所にどんどん溜まり、やがて上流もキューが満杯になって止まります。
**エラーは出ません。ただ画面が固まります。**

対処には、いくつかの考え方があります。

**① 捨てるときに「捨てた」と伝える**

いちばん筋がよい方法です。捨てるかわりに「中身なし」の印を付けたフレームを流せば、
並べ直しは正常に進みます。**欠番を作らない**のが根本的な解決です。

**② 預かりが増えすぎたら、あきらめる**

「預かりが N 枚を超えたら、いちばん若い番号はもう来ないものとして先へ進む」。
追い越しの幅には上限があるので（発展課題1）、その上限より大きい N を選べば、
正常なフレームを取りこぼすことはありません。

**③ 時間で区切る**

「`next` 番を待ち始めてから 100ms 経ったらあきらめる」。
考え方は②と同じで、基準が枚数か時間かの違いです。

次のセルで、②を実装した版と、何もしない版を比べます。
何もしない版は止まるので、**8秒で強制終了**させます。

In [ ]:
%%writefile ans08b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <map>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（ここでは中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 30;
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }
int infer_ms(int i) { return 20 + (i % 3) * 40; }   // 20 / 60 / 100、平均 60

const int LOST = 5;          // 5番のフレームだけ、推論の途中で捨てられたことにする

void run(bool give_up) {
    BoundedQueue<int> q1(4), q2(4);
    std::vector<int> shown;

    std::thread reader([&] { for (int i = 0; i < N; i++) { wait_ms(10); q1.push(i); } });
    std::vector<std::thread> inferers;
    for (int k = 0; k < 2; k++)
        inferers.emplace_back([&, k] {
            for (int i = k; i < N; i += 2) {
                int f = q1.pop();
                wait_ms(infer_ms(f));
                if (f == LOST) continue;               // ← q2 に入れずに捨てる
                q2.push(f);
            }
        });

    std::thread shower([&] {
        std::map<int, int> pending;
        int next = 0;
        while ((int)shown.size() < N - 1) {            // 捨てた1枚を除いて全部出したら終わり
            pending[q2.pop()] = 1;                     // ← ここで永久に待つことになる

            // あきらめる版：預かりが増えすぎたら、いちばん小さい番号はもう来ないと判断する
            if (give_up && pending.size() > 4) next = pending.begin()->first;

            while (!pending.empty() && pending.begin()->first == next) {
                shown.push_back(next);
                pending.erase(pending.begin());
                wait_ms(30);
                next++;
            }
        }
    });

    reader.join();
    for (auto& t : inferers) t.join();
    std::cout << "  （推論はすべて終わった。ここまでに表示できたのは "
              << shown.size() << " 枚）\n" << std::flush;
    shower.join();
    std::cout << "  表示できた枚数 = " << shown.size() << " / " << (N - 1) << "\n\n" << std::flush;
}

int main() {
    std::cout << LOST << " 番のフレームが、推論の途中で捨てられたとする\n\n";

    std::cout << "【あきらめる版】預かりが増えすぎたら、その番号は来ないと判断する\n" << std::flush;
    run(true);

    std::cout << "【素直に待つ版】" << LOST << " 番が来るまで待ち続ける\n" << std::flush;
    run(false);

    std::cout << "ここには到達しない\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans08b.cpp -o ans08b
!timeout 8 ./ans08b; echo "終了コード=$? （124 なら止まった）"

```
【あきらめる版】
  （推論はすべて終わった。ここまでに表示できたのは 28 枚）
  表示できた枚数 = 29 / 29

【素直に待つ版】
  （推論はすべて終わった。ここまでに表示できたのは 5 枚）
  → ここで止まる
```

**素直に待つ版は、5番が捨てられた時点で 5枚しか出せていません。**
推論はとっくに全部終わっているのに、6番以降が1枚も出せていない。
これが「静かに止まる」ということです。

**注意点が1つ**あります。②の方法は「**追い越しの幅の上限**」を知っていることが前提です。
上限より小さい枚数であきらめると、**まだ来る予定のフレームを捨ててしまいます。**
発展課題1で測った「預かる最大数」が、その根拠になります。

> **あきらめる仕組みを入れるなら、あきらめる基準を測って決める。**

## 発展課題3 の解答 ―― 配列（リングバッファ）で作れるか

**作れます。** 条件は1つです。

> **追い越しの幅に、はっきりした上限があること。**

上限が K 枚と分かっていれば、長さ K 以上の配列を用意して
`pending[id % K]` に置けば済みます。`std::map` は要りません。

```cpp
Frame slot[K];
bool  filled[K] = {false};
int   next = 0;

Frame f = q2.pop();
slot[f.id % K] = f; filled[f.id % K] = true;

while (filled[next % K]) {
    出す(slot[next % K]);
    filled[next % K] = false;
    next++;
}
```

**利点**は、メモリの確保が一切起きないことです。`std::map` は要素ごとにメモリを取ります。
1フレームが数百KBあるような処理では、この差は無視できません。

**危険**は、上限を超えたときに**黙って壊れる**ことです。
`id % K` が衝突して、まだ出していないフレームを上書きしてしまいます。

`std::map` は上限を超えても壊れず、増えるだけです（そのかわり気づきにくい）。

> **リングバッファは「上限を知っている」ことと引き換えに速い。**
> **上限を知らないなら `std::map` のほうが安全。**

まずは `std::map` で書き、預かり枚数を測ってから置き換える、というのが安全な順序です。

## 発展課題4 の解答 ―― 並べ直し専用のスレッドにする

```
いま     : Infer ──q2──▶ [ 並べ直し + Show ]
専用化後 : Infer ──q2──▶ [ 並べ直し ] ──q3──▶ [ Show ]
```

**良くなること**

- **段の役割がはっきりします。** Show は「順番に来る」ことだけを前提にでき、
  並べ直しのコードを知らなくてよくなります
- **Show の中身が重い場合、並べ直しと表示が並行に動けます。**
  いまの実装では、Show が 30ms 表示しているあいだ `q2` から取り出せていません

**悪くなること**

- **段が1つ増えるので、レイテンシが増えます**（演習7の発展課題6）
- **キューが1本増えます。** メモリも、出し入れの手間も増えます
- **スレッドが1本増えます。** 待ち中心の仕事なのでコアは食いませんが、無料ではありません

**判断**

並べ直し自体はほとんど時間を使いません（`map` に入れて出すだけ）。
**その程度の処理のために段を1つ増やすのは、たいてい割に合いません。**

ただし「並べ直しの結果を複数の下流に配りたい」「Show が2人いる」といった場合には、
独立させるほうが素直になります。

## 発展課題5 の解答 ―― 担当ごとにレーンを分ける方式

```
案A（8-2）   Read ──▶ [1本のキュー] ──▶ Infer係A/B ──▶ [1本のキュー] ──▶ 並べ直し ──▶ Show
案B（レーン） Read ──▶ [Aの入口] ──▶ Infer係A ──▶ [Aの出口] ──┐
                └──▶ [Bの入口] ──▶ Infer係B ──▶ [Bの出口] ──┴──▶ Show（交互に回収）
```

案Bは並べ直しが要りません。Read が「A,B,A,B...」と配り、Show が「A,B,A,B...」と
回収すれば、順番は自動的に保たれます。

**では、どちらが速いのでしょうか。答えは「中身による」です。** 次のセルで測ります。

In [ ]:
%%writefile ans08c.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <map>
#include <memory>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（ここでは中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 30;
const int K = 2;                                   // Infer の人数
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }
int pattern = 0;
// pattern 0 : 20 / 60 / 100 が順に来る（3フレーム周期。2人に均等に散らばる）
// pattern 1 : 偶数フレームだけ重い（100ms）、奇数は軽い（20ms）
int infer_ms(int i) {
    if (pattern == 0) return 20 + (i % 3) * 40;
    return (i % 2 == 0) ? 100 : 20;
}

// 案A：みんなで1本のキューを取り合い、出口で並べ直す（演習8-2 の方法）
int plan_reorder() {
    BoundedQueue<int> q1(4), q2(4);
    auto t0 = steady_clock::now();
    std::thread reader([&] { for (int i = 0; i < N; i++) { wait_ms(10); q1.push(i); } });
    std::vector<std::thread> inferers;
    for (int k = 0; k < K; k++)
        inferers.emplace_back([&, k] {
            for (int i = k; i < N; i += K) { int f = q1.pop(); wait_ms(infer_ms(f)); q2.push(f); }
        });
    std::thread shower([&] {
        std::map<int, int> pending; int next = 0;
        for (int i = 0; i < N; i++) {
            pending[q2.pop()] = 1;
            while (!pending.empty() && pending.begin()->first == next) {
                pending.erase(pending.begin()); wait_ms(30); next++;
            }
        }
    });
    reader.join();
    for (auto& t : inferers) t.join();
    shower.join();
    return duration_cast<milliseconds>(steady_clock::now() - t0).count();
}

// 案B：担当ごとに入口も出口も分け、順番に配って順番に回収する（並べ直し不要）
int plan_lanes() {
    std::vector<std::unique_ptr<BoundedQueue<int>>> in, out;
    for (int k = 0; k < K; k++) {
        in .push_back(std::make_unique<BoundedQueue<int>>(4));
        out.push_back(std::make_unique<BoundedQueue<int>>(4));
    }
    auto t0 = steady_clock::now();
    std::thread reader([&] {
        for (int i = 0; i < N; i++) { wait_ms(10); in[i % K]->push(i); }   // 交互に配る
    });
    std::vector<std::thread> inferers;
    for (int k = 0; k < K; k++)
        inferers.emplace_back([&, k] {
            for (int i = k; i < N; i += K) { int f = in[k]->pop(); wait_ms(infer_ms(f)); out[k]->push(f); }
        });
    std::thread shower([&] {
        for (int i = 0; i < N; i++) { out[i % K]->pop(); wait_ms(30); }    // 交互に回収する
    });
    reader.join();
    for (auto& t : inferers) t.join();
    shower.join();
    return duration_cast<milliseconds>(steady_clock::now() - t0).count();
}

void compare() {
    int a = plan_reorder(), b = plan_lanes();
    std::cout << std::fixed << std::setprecision(1);
    std::cout << "  1本のキュー + 出口で並べ直す : " << a << " ms   " << (1000.0 * N / a) << " FPS\n";
    std::cout << "  担当ごとにレーンを分ける     : " << b << " ms   " << (1000.0 * N / b) << " FPS\n\n";
}

int main() {
    std::cout << "どちらも順番は正しく保たれる。速さだけを比べる。\n";
    std::cout << "どちらの Infer も、平均は 60ms/フレームで同じ。\n\n";

    std::cout << "【1】重さが 20/60/100 と順に来る場合（2人に均等に散らばる）\n";
    pattern = 0; compare();

    std::cout << "【2】偶数フレームだけ重い場合（100ms と 20ms が交互）\n";
    pattern = 1; compare();
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans08c.cpp -o ans08c && ./ans08c

```
【1】重さが 20/60/100 と順に来る場合（2人に均等に散らばる）
  1本のキュー + 出口で並べ直す : 1063 ms   28.2 FPS
  担当ごとにレーンを分ける     :  974 ms   30.8 FPS

【2】偶数フレームだけ重い場合（100ms と 20ms が交互）
  1本のキュー + 出口で並べ直す : 1015 ms   29.6 FPS
  担当ごとにレーンを分ける     : 1573 ms   19.1 FPS
```

**Infer の平均はどちらの場合も 60ms で同じ**です。それでも結果はまったく違います。

- **【1】仕事が均等に散らばる場合** ⇒ レーン方式のほうがわずかに速い。
  並べ直しの手間がないぶん得をしています
- **【2】偏りがある場合** ⇒ **レーン方式は 19.1 FPS まで落ちます**

【2】で何が起きているか。Read が交互に配るので、
**重いフレームは全部A、軽いフレームは全部B**に行きます。
Aは 100ms ずつ、Bは 20ms ずつ。そして Show は A,B,A,B と順番に取りに行くので、
**毎回Aを待つことになります。** Bは暇なのに、手伝えません。

一方、案Aの1本のキューでは、**手が空いた人が次を取ります。**
誰が重いフレームを引くかは決まっておらず、**勝手に均されます。**

> **1本のキューを取り合う形は、それ自体が仕事の均し役になっている。**

これは演習7-3 で「キューがそのまま仕事の配り口になる」と書いたことの、
もう一歩先の話です。**動的に配ることの価値は、順序を犠牲にしてでも取りに行く価値がある**
ということです。

（そして、その犠牲は 8-2 の並べ直しで取り返せます。）

なお、実際の映像で「偶数フレームだけ重い」ことは稀ですが、
**シーンが切り替わって数秒間だけ重くなる**ことはよくあります。
レーン方式では、その数秒がまるごと片方の担当者に乗ります。

## 発展課題6 の解答 ―― 順序が崩れて困る段・困らない段

段ごとに「崩れると何が困るか」を書き出すと、こうなります。

**Read（読み込み）**

順序は崩しようがありません。ファイルや映像は前から順に読むしかないからです。
**そもそも並列化できない段**です（並列化しても、読む順番を決められません）。

**Infer（推論）**

**崩れます。そして、崩れて構いません。** ―― ただし条件が1つあります。

> **フレームが自分の番号を持ち歩いていること。**

推論結果がどの順で出てこようと、`{id, box}` の組で運ばれていれば、
情報としては何も失われていません。あとで並べ直せます。

逆に、番号を持たせず「来た順に処理する」前提で書かれていると、
8-1 で見たとおり**結果が別のフレームに貼り付きます。**

**Show（表示）**

**崩れてはいけません。** 画面に出す順番が、そのまま見た目になります。
ここが「順序を回復しなければならない場所」です。

### まとめると

```
Read   : 並列化できない        → 順序は崩れない
Infer  : 並列化したい          → 崩れる。番号を持たせて、あとで直す
Show   : 順序が意味を持つ      → 直った状態で受け取る
```

**「崩れてよい区間」と「崩れてはいけない場所」を分ける**のが設計です。
全部の段で順序を守ろうとすると、並列化そのものができなくなります。

> **順序は、必要な場所でだけ回復すればよい。**